# Preprocessing (Updated for `light_curves.csv`)
This notebook cleans ZTF-style light curve observations and produces:
- `light_curves_clean.csv`: cleaned observation table
- `light_curves_sequences.npz`: fixed-length per-object sequences for contrastive learning

**Input columns (expected):** `oid, mjd, fid, mag, e_mag, magpsf, sigmapsf, ra, dec, isdiffpos`

In [8]:
import os
import numpy as np
import pandas as pd

DATA_PATH = 'light_curves.csv'  # <-- new dataset
OUT_DIR = 'preprocessed'
os.makedirs(OUT_DIR, exist_ok=True)
# Ensure outputs are written into the workspace folder (absolute path)
WORKSPACE_ROOT = r'c:/Users/NIPUN/Desktop/LIGHTNING'
os.makedirs(os.path.join(WORKSPACE_ROOT, OUT_DIR), exist_ok=True)
clean_csv_path = os.path.join(WORKSPACE_ROOT, OUT_DIR, 'light_curves_clean.csv')
seq_npz_path = os.path.join(WORKSPACE_ROOT, OUT_DIR, 'light_curves_sequences.npz')

# Load
obs = pd.read_csv(DATA_PATH)
print('Loaded:', DATA_PATH)
print('Shape:', obs.shape)
print('Columns:', list(obs.columns))
obs.head()

Loaded: light_curves.csv
Shape: (20000, 10)
Columns: ['oid', 'mjd', 'fid', 'mag', 'e_mag', 'magpsf', 'sigmapsf', 'ra', 'dec', 'isdiffpos']


,oid,mjd,fid,mag,e_mag,magpsf,sigmapsf,ra,dec,isdiffpos
0,ZTF18aazeojq,58278.407130,1,NaN,NaN,16.692076,0.025775,307.792636,51.134943,-1
1,ZTF18aazeojq,58281.403681,1,NaN,NaN,16.631727,0.024336,307.792558,51.134826,-1
2,ZTF18aazeojq,58285.413102,2,NaN,NaN,16.383300,0.027792,307.792625,51.134461,1
3,ZTF18aazeojq,58287.404167,1,NaN,NaN,16.658768,0.025382,307.792841,51.134746,-1
4,ZTF18aazeojq,58288.407836,1,NaN,NaN,16.912527,0.146301,307.792625,51.135020,1


## 1) Standardize columns
We will use **PSF magnitude** for modeling:
- `mag_used` = `magpsf` (fallback to `mag` if needed)
- `err_used` = `sigmapsf` (fallback to `e_mag` if needed)

In [9]:
obs = obs.copy()

# normalize placeholders and infinities
obs.replace(['', ' ', 'NA', 'NaN', 'nan', 'None', 'none', 'NULL'], np.nan, inplace=True)
obs.replace([np.inf, -np.inf], np.nan, inplace=True)

# choose magnitude + error columns robustly
obs['mag_used'] = obs['magpsf']
obs.loc[obs['mag_used'].isna(), 'mag_used'] = obs.loc[obs['mag_used'].isna(), 'mag']

obs['err_used'] = obs['sigmapsf']
obs.loc[obs['err_used'].isna(), 'err_used'] = obs.loc[obs['err_used'].isna(), 'e_mag']

# keep only the columns we will use downstream
keep_cols = ['oid','mjd','fid','ra','dec','isdiffpos','mag_used','err_used']
missing = [c for c in keep_cols if c not in obs.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
obs = obs[keep_cols].copy()

print('After standardization:', obs.shape)
obs.head()

After standardization: (20000, 8)


,oid,mjd,fid,ra,dec,isdiffpos,mag_used,err_used
0,ZTF18aazeojq,58278.407130,1,307.792636,51.134943,-1,16.692076,0.025775
1,ZTF18aazeojq,58281.403681,1,307.792558,51.134826,-1,16.631727,0.024336
2,ZTF18aazeojq,58285.413102,2,307.792625,51.134461,1,16.383300,0.027792
3,ZTF18aazeojq,58287.404167,1,307.792841,51.134746,-1,16.658768,0.025382
4,ZTF18aazeojq,58288.407836,1,307.792625,51.135020,1,16.912527,0.146301


## 2) Basic cleaning
- drop duplicates
- drop rows missing critical fields (`oid, mjd, fid, mag_used`)
- enforce dtypes
- sanity filter RA/Dec ranges

In [10]:
# drop duplicates
before = len(obs)
obs = obs.drop_duplicates().reset_index(drop=True)
print('Dropped duplicates:', before - len(obs))

# drop rows missing critical fields
critical = ['oid','mjd','fid','mag_used']
before = len(obs)
obs = obs.dropna(subset=critical).reset_index(drop=True)
print('Dropped rows with missing critical fields:', before - len(obs))

# dtypes
obs['oid'] = obs['oid'].astype(str)
obs['mjd'] = pd.to_numeric(obs['mjd'], errors='coerce')
obs['fid'] = pd.to_numeric(obs['fid'], errors='coerce').astype('Int64')
obs['ra'] = pd.to_numeric(obs['ra'], errors='coerce')
obs['dec'] = pd.to_numeric(obs['dec'], errors='coerce')
obs['isdiffpos'] = pd.to_numeric(obs['isdiffpos'], errors='coerce').fillna(0).astype(int)
obs['mag_used'] = pd.to_numeric(obs['mag_used'], errors='coerce')
obs['err_used'] = pd.to_numeric(obs['err_used'], errors='coerce')

# sanity filter RA/Dec
if 'ra' in obs.columns:
    before = len(obs)
    obs = obs[(obs['ra'] >= 0) & (obs['ra'] <= 360)].reset_index(drop=True)
    print('Filtered RA outside [0,360]:', before - len(obs))
if 'dec' in obs.columns:
    before = len(obs)
    obs = obs[(obs['dec'] >= -90) & (obs['dec'] <= 90)].reset_index(drop=True)
    print('Filtered Dec outside [-90,90]:', before - len(obs))

# sort
obs = obs.sort_values(['oid','mjd','fid']).reset_index(drop=True)
print('Rows after cleaning:', len(obs))
obs.head()

Dropped duplicates: 264
Dropped rows with missing critical fields: 0
Filtered RA outside [0,360]: 0
Filtered Dec outside [-90,90]: 0
Rows after cleaning: 19736


,oid,mjd,fid,ra,dec,isdiffpos,mag_used,err_used
0,ZTF18aajtltx,58252.436134,1,274.382237,58.493638,-1,19.838300,0.104995
1,ZTF18aajtltx,58252.443218,1,274.382248,58.493561,-1,19.928700,0.133612
2,ZTF18aajtltx,58252.491157,2,274.382250,58.493683,-1,19.125100,0.137872
3,ZTF18aajtltx,58261.453738,1,274.382241,58.493638,-1,20.198800,0.143248
4,ZTF18aajtltx,58272.391377,1,274.382412,58.493586,-1,19.766142,0.178495


## 3) Outlier clipping (CL-friendly)
We clip extreme values for `mag_used` and `err_used` (not RA/Dec) to robust percentiles to avoid breaking rare patterns.

In [11]:
def clip_series(s: pd.Series, lo_q=0.01, hi_q=0.99) -> pd.Series:
    lo = s.quantile(lo_q)
    hi = s.quantile(hi_q)
    if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
        return s.clip(lo, hi)
    return s

for col in ['mag_used','err_used']:
    if col in obs.columns:
        before_min, before_max = obs[col].min(), obs[col].max()
        obs[col] = clip_series(obs[col])
        after_min, after_max = obs[col].min(), obs[col].max()
        print(col, 'min/max:', (before_min, before_max), '->', (after_min, after_max))

mag_used min/max: (np.float64(13.544434), np.float64(20.5377)) -> (np.float64(14.91790455), np.float64(19.37839))
err_used min/max: (np.float64(0.009013123), np.float64(0.499309)) -> (np.float64(0.011751), np.float64(0.2305998500000009))


## 4) Save cleaned observation table

In [12]:
obs.to_csv(clean_csv_path, index=False)
print('Saved cleaned CSV ->', clean_csv_path)
obs.head()

Saved cleaned CSV -> c:/Users/NIPUN/Desktop/LIGHTNING\preprocessed\light_curves_clean.csv


,oid,mjd,fid,ra,dec,isdiffpos,mag_used,err_used
0,ZTF18aajtltx,58252.436134,1,274.382237,58.493638,-1,19.37839,0.104995
1,ZTF18aajtltx,58252.443218,1,274.382248,58.493561,-1,19.37839,0.133612
2,ZTF18aajtltx,58252.491157,2,274.382250,58.493683,-1,19.12510,0.137872
3,ZTF18aajtltx,58261.453738,1,274.382241,58.493638,-1,19.37839,0.143248
4,ZTF18aajtltx,58272.391377,1,274.382412,58.493586,-1,19.37839,0.178495


## 5) Build fixed-length per-object sequences
For contrastive learning, we often want a **consistent length** per object.

We will:
1. group by `oid`
2. normalize time per object: `t = (mjd - min_mjd)`
3. create a uniform time grid of length `L`
4. for each `fid` band, interpolate `mag_used` and `err_used` onto the grid
5. produce arrays:
   - `X_mag`: shape `[N, L, B]`
   - `X_err`: shape `[N, L, B]`
   - `X_mask`: shape `[N, L, B]` (1 where real observations existed nearby)
   - `t_grid`: shape `[L]`

This is a strong starting format for SimCLR/BYOL augmentations.

In [13]:
from collections import defaultdict

# Parameters (change as desired)
L = 50  # fixed sequence length
min_points_per_oid = 5  # minimum raw observations per object
seeing_threshold = 5.0
airmass_threshold = 3.0
require_isdiffpos = True  # drop rows where isdiffpos == 0 if column exists
rng = np.random.default_rng(42)

# Use magpsf/sigmapsf when available; fall back to mag_used/err_used
mag_col = 'magpsf' if 'magpsf' in obs.columns else 'mag_used'
err_col = 'sigmapsf' if 'sigmapsf' in obs.columns else 'err_used'
print('Using columns:', mag_col, err_col)

# 1-4) Additional cleaning per user request
before = len(obs)
# Drop rows missing key cols
key_cols = ['oid','mjd','fid', mag_col, err_col]
obs = obs.dropna(subset=key_cols).reset_index(drop=True)
print('Dropped missing key cols:', before - len(obs))

# Drop invalid uncertainties and magnitudes
before = len(obs)
obs = obs[obs[err_col].astype(float) > 0].copy()
obs = obs[(obs[mag_col].astype(float) >= 0) & (obs[mag_col].astype(float) <= 30)].copy()
print('Dropped invalid sig/mag rows:', before - len(obs))

# Drop duplicate (oid,mjd,fid)
before = len(obs)
obs = obs.drop_duplicates(subset=['oid','mjd','fid']).reset_index(drop=True)
print('Dropped duplicate (oid,mjd,fid):', before - len(obs))

# Optional quality filters (if columns exist)
if 'seeing' in obs.columns:
    before = len(obs)
    obs = obs[obs['seeing'].astype(float) <= seeing_threshold].copy()
    print('Filtered by seeing:', before - len(obs))
if 'airmass' in obs.columns:
    before = len(obs)
    obs = obs[obs['airmass'].astype(float) <= airmass_threshold].copy()
    print('Filtered by airmass:', before - len(obs))
if require_isdiffpos and 'isdiffpos' in obs.columns:
    before = len(obs)
    obs = obs[obs['isdiffpos'].astype(int) != 0].copy()
    print('Filtered isdiffpos==0:', before - len(obs))

# 5) Convert magnitude -> flux (if desired). We'll compute both mag and flux channels so users can pick.
# flux = 10^(-0.4 * mag); sigma_flux = 0.4 * ln(10) * flux * sigma_mag
obs['mag_val'] = obs[mag_col].astype(float)
obs['mag_err'] = obs[err_col].astype(float)
obs['flux_val'] = 10 ** (-0.4 * obs['mag_val'])
obs['flux_err'] = (0.4 * np.log(10)) * obs['flux_val'] * obs['mag_err']

# Choose feature mode: 'mag' or 'flux'
feature_mode = 'flux'  # change to 'mag' to use magnitudes
val_col = 'flux_val' if feature_mode == 'flux' else 'mag_val'
err_col_use = 'flux_err' if feature_mode == 'flux' else 'mag_err'

# 6) Normalize time per object later during sequence building (t_rel)

# 7) Encode filter band
bands = sorted([b for b in obs['fid'].dropna().unique()])
bands = [int(b) for b in bands]
band2idx = {b: i for i, b in enumerate(bands)}
B = len(bands)
print('Bands (fid):', bands)

# 8-11) Group by oid, build fixed-length grid and interpolate per band
counts = obs.groupby('oid').size()
keep_oids = counts[counts >= min_points_per_oid].index.tolist()
obs_f = obs[obs['oid'].isin(keep_oids)].copy()
print('Objects with >= min points:', len(keep_oids))

def interp_with_mask(t_src, y_src, t_grid):
    t_src = np.asarray(t_src, dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)
    m = np.isfinite(t_src) & np.isfinite(y_src)
    t_src = t_src[m]
    y_src = y_src[m]
    if len(t_src) == 0:
        return np.full_like(t_grid, np.nan, dtype=np.float32), np.zeros_like(t_grid, dtype=np.float32)
    if len(t_src) == 1:
        # single point: constant value and mask around nearest grid cell
        y = np.full_like(t_grid, float(y_src[0]), dtype=np.float32)
        d = np.abs(t_grid - float(t_src[0]))
        win = 0.5  # half-day proximity for single obs
        mask = (d <= win).astype(np.float32)
        return y, mask
    order = np.argsort(t_src)
    t_src = t_src[order]
    y_src = y_src[order]
    y = np.interp(t_grid, t_src, y_src).astype(np.float32)
    span = float(t_src.max() - t_src.min())
    win = max(0.1, 0.01 * (span + 1e-6))
    d = np.min(np.abs(t_grid[:, None] - t_src[None, :]), axis=1)
    mask = (d <= win).astype(np.float32)
    return y, mask

oid_list = []
X_list = []  # will hold (L, B, C) arrays per object
t_grids = []

for oid, g in obs_f.groupby('oid'):
    g = g.sort_values('mjd')
    t0 = float(g['mjd'].min())
    t_rel = (g['mjd'].astype(float) - t0).to_numpy()
    t_span = float(np.nanmax(t_rel) - np.nanmin(t_rel))
    if not np.isfinite(t_span) or t_span <= 0:
        continue
    t_grid = np.linspace(0.0, t_span, L).astype(np.float32)
    val_cube = np.zeros((L, B), dtype=np.float32)
    err_cube = np.zeros((L, B), dtype=np.float32)
    mask_cube = np.zeros((L, B), dtype=np.float32)
    val_cube.fill(np.nan)
    err_cube.fill(np.nan)
    for fid in bands:
        j = band2idx[fid]
        gb = g[g['fid'] == fid]
        if gb.empty:
            continue
        y_val, m_val = interp_with_mask((gb['mjd'].astype(float) - t0).to_numpy(), gb[val_col].to_numpy(), t_grid)
        y_err, _ = interp_with_mask((gb['mjd'].astype(float) - t0).to_numpy(), gb[err_col_use].to_numpy(), t_grid)
        val_cube[:, j] = y_val
        err_cube[:, j] = y_err
        mask_cube[:, j] = m_val
    # require some coverage
    if mask_cube.sum() < 3:
        continue
    # 12) Stack features -> channels mag/flux, err, mask
    # Replace NaNs in val/err with 0 temporarily; mask indicates validity
    val_cube = np.where(np.isfinite(val_cube), val_cube, 0.0).astype(np.float32)
    err_cube = np.where(np.isfinite(err_cube), err_cube, 0.0).astype(np.float32)
    X_obj = np.stack([val_cube, err_cube, mask_cube], axis=-1)  # (L, B, 3)
    oid_list.append(oid)
    X_list.append(X_obj)
    t_grids.append(t_grid)

if len(X_list) == 0:
    print('No objects passed filters; adjust thresholds or min_points_per_oid')
    X = np.zeros((0, L, B, 3), dtype=np.float32)
else:
    X = np.stack(X_list, axis=0)

print('Built tensor X shape (N, L, B, C):', X.shape)

# 13) Normalize per-object: for feature channel 0 (value) normalize per object across time and band where mask==1
if X.shape[0] > 0:
    X_norm = X.copy()
    for i in range(X_norm.shape[0]):
        vals = X_norm[i, :, :, 0]
        mask = X_norm[i, :, :, 2].astype(bool)
        if mask.sum() > 0:
            data = vals[mask]
            mu = data.mean()
            sigma = data.std() if data.std() > 0 else 1.0
            X_norm[i, :, :, 0] = (vals - mu) / sigma
            # scale errors accordingly if using flux; errors are absolute so divide by sigma
            X_norm[i, :, :, 1] = X_norm[i, :, :, 1] / sigma
        else:
            X_norm[i, :, :, 0] = 0.0
            X_norm[i, :, :, 1] = 0.0
else:
    X_norm = X

# 16) Split by oid (80/10/10)
N = X_norm.shape[0]
idxs = np.arange(N)
rng.shuffle(idxs)
n_train = int(0.8 * N)
n_val = int(0.1 * N)
train_idx = idxs[:n_train]
val_idx = idxs[n_train:n_train + n_val]
test_idx = idxs[n_train + n_val:]
splits = {'train': train_idx, 'val': val_idx, 'test': test_idx}
print('Split sizes (train,val,test):', len(train_idx), len(val_idx), len(test_idx))

# 15 & 17) Save NPZ with X, oids, bands, L, and splits
np.savez_compressed(seq_npz_path,
                    oid=np.array(oid_list, dtype=object),
                    bands=np.array(bands, dtype=np.int32),
                    L=np.int32(L),
                    X=X_norm.astype(np.float32),
                    splits=splits,
                    feature_mode=np.array(feature_mode)
                   )
print('Saved sequences NPZ ->', seq_npz_path)

# simple summary of coverage
if X_norm.shape[0] > 0:
    print('Fraction missing (mask==0):', 1.0 - (X_norm[:, :, :, 2].mean()))
    print('Example oid:', oid_list[0])
    print('Per-band coverage for example:', X_norm[0, :, :, 2].sum(axis=0))


Using columns: mag_used err_used
Dropped missing key cols: 0
Dropped invalid sig/mag rows: 0
Dropped duplicate (oid,mjd,fid): 2783
Filtered isdiffpos==0: 0
Bands (fid): [1, 2]
Objects with >= min points: 5
Built tensor X shape (N, L, B, C): (5, 50, 2, 3)
Split sizes (train,val,test): 4 0 1
Saved sequences NPZ -> c:/Users/NIPUN/Desktop/LIGHTNING\preprocessed\light_curves_sequences.npz
Fraction missing (mask==0): 0.102
Example oid: ZTF18aajtltx
Per-band coverage for example: [46. 48.]
